# FFM_3_8 — mesh → latent 在 48 kHz 上重做，按 S2RNF 的口径评估

**2026-10-12 建。** FFM_3_2_1（44.1 kHz，FFM26）的 mesh → latent 在 48 kHz 数据和 `FFM48k_measim` 上重做，
评估口径向 Masuyama et al.（Interspeech 2026，S2RNF）靠拢，方便和他们的结果对照。不追求 3 dB。

## 和之前的区别

| | FFM_3_2_1 / Task 1 | 本 notebook |
|---|---|---|
| 数据 · AE | 44.1 kHz · FFM26（Task 1 已是 48 kHz · measim） | 48 kHz（10-10 修过的数据）· `FFM48k_measim` |
| 特征标准化 | 在全部 164 人上 z-score，再分折 | **每个内 / 外层训练折内部**估计均值、标准差、常数列和 kernel 定标，再用到验证折 |
| 输入（→ 实测 z） | 几何 | 几何、仿真（同一个人的 BEM，S2RNF 的输入）、几何 + 仿真 |
| 参照 | 均值、AE 重建 | 均值、**原始仿真**（S2RNF 表里的 Mesh2HRTF 行）、**仿真偏差 + 实测均值**、AE 重建参考 |
| 主指标 | pooled RMSE（含 DC，全球面） | **LSD 与 MAE，bin 1–85（不含 DC），仰角 ≥ −45°**；全球面 LSD 和旧 pooled RMSE 也报 |
| 统计 | CV-R²、置换、bootstrap | 同上，另加按人配对的单侧 t 检验（S2RNF 的做法）和逐频率误差曲线（S2RNF 图 3） |

几何特征不重新挑：沿用 FFM_3_2_1 选出的 `simple + ear multi-σ` RFF（20 000 个采样点；FFM_3_6 已证明和精确积分差 ≤ 0.003 dB）。
回归器是核 ridge，α 由内层 5 折按人分组 CV、按 latent 误差选（与之前相同）。

## 和 S2RNF 仍然不同的地方（报告里要写）

- 频带：他们到 20 kHz，我们的 AE 只到 15.94 kHz（86 个 bin）。
- 方向：他们是原生 793 个测点；我们是 HEALPix，≥ −45° 的区域也是插值出来的。
- 数据版本：他们用 `*_Windowed_NoITD_Scaled.sofa`，我们用 `FreeFieldCompMinPhase_48kHz` 和 `HRIR_SONICOM_48000`。
- 划分：同样 200 人，但两边划分交叉（他们测试集 25 人中 21 人在我们 AE 训练集；我们锁定 36 人中 31 人在他们训练集），
  所以只能各自相对自己的均值基线比较。测试格里另算一个"协议校准"：在他们的 160 / 25 划分上按我们的口径算均值基线和原始仿真。
- 我们的 AE 还用了其他 SONICOM 受试者和另外三个库预训练。

## 运行

1. **开发集（Cell 0–8）**：先复现 Task 1 的数（全局标准化），复现不了就停；再跑每折内部标准化的正式版本。GPU，约 10–20 分钟。
2. **测试（Cell 9）**：`RUN_TEST = False`。看完开发集、把 Cell 1 的登记内容写进 PROJECT_LOG 之后才改成 True，只跑一次。
   36 个锁定受试者也在 FFM48k 的 SONICOM 测试集里：这一格会读 measim 在这 36 人上的测试结果。
   与 FFM_3_5 / 3_6 / 3_7 共用"已用过"标记，跑过之后它们的测试格都会拒绝。

## Drive 上要有的文件（都是 Task 1 用过的位置）

`ELEC5021/data_48k_2026-10-10.zip`、`ELEC5021/split_48k.json`、`HRTF_Checkpoints/FFM48k_measim_best.pth`、
`ELEC5020/SONICOM/mesh2z_samples.zip`、`HRTF_Checkpoints/mesh2z_sonicom_latents_v1.npz`、`HRTF_Checkpoints/mesh2z_rff_v1/oof_ridge.npz`。
结果写到 `HRTF_Checkpoints/mesh2z_48k_v1/`。

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 数据（zip 按 SHA-256 核对后解压到本地盘；读到的每个数组再按 manifest 核对）
# ==========================================================================
import torch
import os, json, csv, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
from scipy import stats
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
PATHS = dict(
    data_zip    = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    split       = f'{DRIVE}/ELEC5021/split_48k.json',
    ckpt        = f'{SAVE_DIR}/FFM48k_measim_best.pth',
    samples_zip = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples.zip',
    # 下面两个只取受试者名单和折号（与 FFM_3_2_1 / Task 1 同一批人、同一组折）；里面 FFM26 的 latent 和旧预测都不读。
    # 48 kHz 的 latent 在 Cell 6 用 FFM48k_measim 现算
    subject_lists = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
    fold_ref      = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',
)
for k, p in PATHS.items():
    print(f'{k:14s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好（路径与 Task 1 相同）'
ZIP_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
SAMPLES_SHA256 = '8340bdf73be1a8722015e877bcca2957570293484277989f1245eb8d65aa085a'
OUT = f'{SAVE_DIR}/mesh2z_48k_v1'
os.makedirs(OUT, exist_ok=True)
# 锁定的 36 人只能用一次：与 FFM_3_5 / 3_6 / 3_7 共用标记
TEST_MARKS = [f'{SAVE_DIR}/TEST_SPENT_SONICOM_locked36.json', f'{SAVE_DIR}/geometry2z_tvt_v1/TEST_SPENT_mesh.json',
              f'{SAVE_DIR}/geometry2z_tvt_v2/TEST_SPENT_mesh.json', f'{OUT}/TEST_SPENT.json']


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def unzip_checked(zip_path, sha, dest):
    stamp = f'{dest}/.zip_ok_{sha[:16]}'
    if os.path.exists(stamp):
        return
    t0 = time.time()
    assert sha256(zip_path) == sha, f'{zip_path}: SHA-256 不符'
    shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    open(stamp, 'w').write(sha)
    print(f'{os.path.basename(zip_path)} 核对并解压 ({time.time() - t0:.0f}s)')


unzip_checked(PATHS['data_zip'], ZIP_SHA256, '/content/data_48k_local')
unzip_checked(PATHS['samples_zip'], SAMPLES_SHA256, '/content/mesh2z_samples_local')
DATA, SAMPLES = '/content/data_48k_local/data_48k', '/content/mesh2z_samples_local/mesh2z_samples'
assert os.path.exists(f'{SAMPLES}/manifest.csv')
assert sha256(PATHS['split']) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT = json.load(open(PATHS['split']))
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ==========================================================================
# Cell 1 — 设定与登记（★ 跑测试之前写死，并抄进 PROJECT_LOG）
# ==========================================================================
N_BINS, FS = 86, 48000
FREQS = np.arange(N_BINS) * FS / 256
EVAL_BINS = slice(1, N_BINS)              # bin 1–85 = 187.5 Hz – 15.94 kHz，不含 DC（S2RNF 也不含 DC）
EL = 90 - np.degrees(hp.pix2ang(16, np.arange(3072))[0])
MEAS_DIRS = EL >= -45.0                   # 2652 个方向；−45° 以下是外推
ALPHA_REL = list(np.logspace(-4, 5, 37))
N_FOLDS, N_INNER, SEED = 10, 5, 0         # 与 FFM_3_2_1 / Task 1 相同的折
N_PERM, N_BOOT, BS = 100, 2000, 16
SIGMAS, M_RFF = [0.005, 0.01, 0.02, 0.05, 0.1], 256
DM, DS = 2, 4                             # FFM48k_measim 的 FiLM 行：SONICOM 实测、SONICOM 仿真

REGISTRATION = dict(
    encoder='FFM48k_measim_best.pth (frozen)',
    features='FFM_3_2_1 simple + ear multi-sigma RFF (sigma 0.005-0.1 /mm, 256 frequencies each), 20k sampled points',
    rows={'geometry': ['geometry'], 'simulation': ['simulation'], 'geometry + simulation': ['geometry', 'simulation']},
    primary_row='geometry',
    primary_metric='LSD, bins 1-85 (DC excluded), elevation >= -45 deg; per subject = mean of both ears',
    secondary_metrics=['MAE same region', 'LSD full sphere bins 1-85', 'pooled RMSE all bins all directions (old)', 'CV-R2(z)'],
    regressor='kernel ridge (dual), alpha by inner 5-fold subject-grouped CV on latent squared error, 37-point relative grid',
    preprocessing='standardisation, constant-column removal and kernel scaling estimated on the training rows of each fold only',
    references=['training mean (measured)', 'raw simulation', 'simulated deviation + measured training mean', 'AE reconstruction (measured input)'],
    tests='paired over subjects: bootstrap 95% CI of the difference (2000), one-sided paired t-test; model vs mean, combined vs geometry',
    test_set='36 locked SONICOM subjects (AE test partition), evaluated once after the development results are reviewed',
)
RUN_TEST = False   # ★ 只在开发集结果看完、登记写进 PROJECT_LOG 之后改成 True，且只跑一次
print(json.dumps(REGISTRATION, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM48k_Train.ipynb 的模型定义逐字相同（构建时复制进来）。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 载入 FFM48k_measim
# ==========================================================================
ck = torch.load(PATHS['ckpt'], map_location='cpu', weights_only=False)
cfg = ck['config']
assert ck['run_name'] == 'FFM48k_measim' and cfg['num_bins'] == N_BINS and cfg['num_datasets'] == 6
assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256
model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=6,
                             num_bins=N_BINS, tie_encoder_film=True).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
FILM_ID = {d['tag']: d['id'] for d in ck['datasets']}
assert FILM_ID['SONICOM'] == DM and FILM_ID['SONICOM_SIM'] == DS
print(f"FFM48k_measim epoch {ck['epoch'] + 1} | val {ck['val_rmse']:.4f} dB | FiLM 行 {FILM_ID}")
del ck

In [ ]:
# ==========================================================================
# Cell 4 — 回归引擎：核 ridge 的对偶形式（Task 1 Cell 4），kernel 由函数按训练行生成
#
# Kfun(ref) 返回全部行的 kernel，但标准化、常数列筛选和 kernel 定标只用 ref 行（训练折）估计。
# 复现关卡用 Kfun = lambda ref: K_global（Task 1 的做法：在全部开发行上估计）。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]


def make_plan(ns, seed=SEED):
    # FFM_3_2_1 的 build_plan：随机数的调用顺序相同，所以折相同
    rng = np.random.default_rng(seed)
    subj = np.repeat(np.arange(ns), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    plan = []
    for te in subject_folds(np.arange(ns), N_FOLDS, rng):
        tr = np.setdiff1d(np.arange(ns), te)
        inner = [(rows(np.setdiff1d(tr, v)), rows(v)) for v in subject_folds(tr, N_INNER, rng)]
        plan.append(((rows(tr), rows(te)), inner))
    return plan


def centred_pair(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot


def centred_vv_diag(G, tr, va):
    r = G[np.ix_(va, tr)].mean(1)
    return np.diag(G[np.ix_(va, va)]) - 2 * r + G[np.ix_(tr, tr)].mean()


def weights(K, tr, va, alphas):
    Ktt, Kvt = centred_pair(K, tr, va)
    lam, Q = np.linalg.eigh(Ktt)
    lam = np.clip(lam, 0, None)
    KvQ, scale = Kvt @ Q, np.trace(Ktt) / len(tr)
    return [(KvQ / (lam + a * scale)) @ Q.T for a in alphas]


def row_sse(W, G, tr, va):
    CYtt, CYvt = centred_pair(G, tr, va)
    return centred_vv_diag(G, tr, va) - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)


def run_cv(Kfun, Y, plan, keep_pred=True):
    G = Y @ Y.T
    n = len(Y)
    sse, sst = np.zeros(n), np.zeros(n)
    pred = np.zeros_like(Y) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, G, itr, iva).sum()
        j = int(np.argmin(tot)); chosen.append(j)
        W = weights(Kfun(tr), tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, G, tr, va); sst[va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            my = Y[tr].mean(0)
            pred[va] = my + W @ (Y[tr] - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, alpha_idx=chosen, pred=pred)


def block_gram(X, ref):
    # 一个特征块：只用 ref 行估计均值、标准差和常数列；除以列数（多块拼接时每块权重相同）
    mu, sd = X[ref].mean(0), X[ref].std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    Xz = (X[:, keep] - mu[keep]) / sd[keep]
    return Xz @ Xz.T / Xz.shape[1]


def unit_scaled(K, ref):
    # kernel 定标：ref 行上中心化后的平均对角元 = 1（Task 1 的 unit_trace 只在 ref 上算）
    n = len(ref); C = np.eye(n) - 1.0 / n
    return K / (np.trace(C @ K[np.ix_(ref, ref)] @ C) / n)


def source_kernel(blocks, ref):
    return unit_scaled(sum(block_gram(X, ref) for X in blocks.values()), ref)


def kernel_fn(sources, fold_internal):
    if fold_internal:
        return lambda ref: sum(source_kernel(S, ref) for S in sources)
    allrows = np.arange(len(next(iter(sources[0].values()))))
    K = sum(source_kernel(S, allrows) for S in sources)
    return lambda ref: K


def fit_predict(Kfun, Y_dev, n_dev, n_all, seed=SEED):
    # 测试用：α 由开发集上的 5 折按人分组 CV 选，再用全部开发行拟合，预测其余行
    G = Y_dev @ Y_dev.T
    rng = np.random.default_rng(seed); subj = np.repeat(np.arange(n_dev // 2), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    tot = np.zeros(len(ALPHA_REL))
    for v in subject_folds(np.arange(n_dev // 2), N_INNER, rng):
        itr, iva = rows(np.setdiff1d(np.arange(n_dev // 2), v)), rows(v)
        for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
            tot[j] += row_sse(W, G, itr, iva).sum()
    j = int(np.argmin(tot))
    dev, te = np.arange(n_dev), np.arange(n_dev, n_all)
    W = weights(Kfun(dev), dev, te, [ALPHA_REL[j]])[0]
    my = Y_dev.mean(0)
    return my + W @ (Y_dev - my), j


# 自检：随机特征的 CV-R² 应在 0 附近或为负；两种标准化都要过
_plan = make_plan(40)
_Y = np.random.default_rng(3).standard_normal((80, 50)); _S = [{'x': np.random.default_rng(4).standard_normal((80, 30))}]
for fi in (False, True):
    print(f"自检（fold_internal={fi}）：随机特征 CV-R² = {run_cv(kernel_fn(_S, fi), _Y, _plan, keep_pred=False)['r2']:+.4f}（应 ≤ ~0）")

In [ ]:
# ==========================================================================
# Cell 5 — 数据 · latent · 几何特征 · 指标
# ==========================================================================
def load_rows(tag, ids):
    # 每人两行（左耳, 镜像后的右耳）；每个数组按 manifest 核对 SHA-256
    d = f'{DATA}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), N_BINS, 3072), np.float32)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[:N_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


def encode(X, did):
    Z = []
    for b in range(0, len(X), BS):
        x = torch.from_numpy(X[b:b + BS]).to(device)
        _, z = model(x, torch.full((len(x),), did, dtype=torch.long, device=device))
        Z.append(z.reshape(len(x), -1).double().cpu().numpy())
    return np.concatenate(Z)


def geometry_blocks(sids):
    # FFM_3_2_1 Cell 4/5（= Task 1 sonicom_geometry），只返回原始特征块，不做标准化
    man = {f"SON_{r['subject']}": r for r in csv.DictReader(open(f'{SAMPLES}/manifest.csv'))}
    flip = np.array([1.0, -1.0, 1.0])
    G0T = torch.as_tensor(np.random.default_rng(12345).standard_normal((M_RFF, 3)).T, dtype=torch.float32, device=device)

    def rff(P):
        base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0T)
        return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
                for sg in SIGMAS}

    pct = lambda a, q: float(np.percentile(a, q))
    simple, ear = [], []
    for s in sids:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'][:20000].astype(np.float64)
        head = head[head[:, 2] > -75.0]
        for side in (0, 1):
            if side == 0:
                pts, canal, area, hd = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(man[s]['crop_area_L_mm2']), head
            else:
                pts, canal, area, hd = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * flip, float(man[s]['crop_area_R_mm2']), head * flip
            assert canal[1] > 0
            q = pts[:20000].astype(np.float64) - canal
            out = q[q[:, 1] > 3.0]
            simple.append([canal[0], canal[1], canal[2], pct(q[:, 1], 95),
                           pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5),
                           area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)])
            ear.append(rff(q))
    blocks = {'simple': np.array(simple)}
    for sg in SIGMAS:
        blocks[f'ear_s{sg}'] = np.stack([f[sg] for f in ear])
    return blocks


M45 = torch.as_tensor(MEAS_DIRS, device=device)


def metrics(pred, X):
    """pred(b0, b1) -> (b, 86, 3072) 张量；X 是实测（numpy）。返回逐耳的各项指标"""
    out = {k: [] for k in ('se_all', 'lsd_el45', 'lsd_full', 'mae_el45', 'mae_full', 'sd_bin_el45', 'sd_bin_full')}
    for b0 in range(0, len(X), BS):
        b1 = min(b0 + BS, len(X))
        e = (pred(b0, b1).float() - torch.from_numpy(X[b0:b1]).to(device)).double()
        out['se_all'].append((e ** 2).sum((1, 2)))                    # 旧口径：全部 bin、全部方向
        eb = e[:, EVAL_BINS]
        d = torch.sqrt((eb ** 2).mean(1))                             # 每个方向沿频率 RMS
        out['lsd_full'].append(d.mean(1)); out['lsd_el45'].append(d[:, M45].mean(1))
        a = eb.abs()
        out['mae_full'].append(a.mean((1, 2))); out['mae_el45'].append(a[:, :, M45].mean((1, 2)))
        out['sd_bin_el45'].append(torch.sqrt((eb[:, :, M45] ** 2).mean(2)))
        out['sd_bin_full'].append(torch.sqrt((eb ** 2).mean(2)))
    return {k: torch.cat(v).cpu().numpy() for k, v in out.items()}


def per_subject(m):
    ns = len(m['se_all']) // 2
    S = {k: m[k].reshape(ns, 2).mean(1) for k in ('lsd_el45', 'lsd_full', 'mae_el45', 'mae_full')}
    S['rmse_all'] = np.sqrt(m['se_all'].reshape(ns, 2).sum(1) / (2 * N_BINS * 3072))
    return S


def summary(m):
    S = per_subject(m)
    r = {k: float(v.mean()) for k, v in S.items() if k != 'rmse_all'}
    r.update({f'sd_{k}': float(S[k].std(ddof=1)) for k in ('lsd_el45', 'mae_el45')})
    r['pooled_rmse'] = float(np.sqrt(m['se_all'].sum() / (len(m['se_all']) * N_BINS * 3072)))
    r['curve_el45'] = m['sd_bin_el45'].mean(0).tolist(); r['curve_full'] = m['sd_bin_full'].mean(0).tolist()
    return r


def decoder_pred(Zrows, did):
    return lambda b0, b1: model.decoder(torch.from_numpy(Zrows[b0:b1].reshape(-1, N_BINS, 64)).float().to(device),
                                        torch.full((b1 - b0,), did, dtype=torch.long, device=device))


def ae_pred(X, did):
    return lambda b0, b1: model(torch.from_numpy(X[b0:b1]).to(device),
                                torch.full((b1 - b0,), did, dtype=torch.long, device=device))[0]


def paired(a, b, rng):
    """a − b over subjects（负 = a 更好）：均值、bootstrap 95% CI、单侧配对 t 检验 p（H1: a < b）"""
    d = a - b
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)])
    return dict(diff=float(d.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
                t_p_one_sided=float(stats.ttest_rel(a, b, alternative='less').pvalue))


MASUYAMA = {'Avg. HRTF': dict(mae=3.78, lsd=5.05), 'Mesh2HRTF (raw simulation)': dict(mae=7.50, lsd=10.53),
            'S2RNF': dict(mae=3.60, lsd=4.90)}     # 他们的表 1（SONICOM 25 人测试集；"RMSE" 即 LSD，到 20 kHz、不含 DC、793 个测点）


def show(title, rows, mean_key='training mean'):
    print(f'\n{title}')
    print(f"{'':<44}{'R²(z)':>8}{'LSD≥−45°':>10}{'vs均值':>8}{'MAE≥−45°':>10}{'LSD全球':>9}{'vs均值':>8}{'旧pooled':>10}")
    m0 = rows[mean_key]
    for k, r in rows.items():
        rel = lambda key: f"{(m0[key] - r[key]) / m0[key]:+8.1%}"
        r2 = f"{r['r2']:+8.4f}" if r.get('r2') is not None else f"{'':>8}"
        print(f"{k:<44}{r2}{r['lsd_el45']:10.3f}{rel('lsd_el45')}{r['mae_el45']:10.3f}{r['lsd_full']:9.3f}{rel('lsd_full')}{r['pooled_rmse']:10.3f}")
    print('  参照（Masuyama 2026 表 1，口径不同，只比"相对均值"）：' +
          '；'.join(f"{k} LSD {v['lsd']:.2f} / MAE {v['mae']:.2f}" for k, v in MASUYAMA.items()) +
          f"；S2RNF 相对均值 LSD {(5.05 - 4.90) / 5.05:+.1%}，MAE {(3.78 - 3.60) / 3.78:+.1%}")

In [ ]:
# ==========================================================================
# Cell 6 — 开发集：受试者、折、数据、latent、几何
# ==========================================================================
L = np.load(PATHS['subject_lists'])          # 只用 eval_sids / locked_sids
DEV = [str(s) for s in L['eval_sids']]; LOCKED = [str(s) for s in L['locked_sids']]
assert len(DEV) == 164 and len(LOCKED) == 36 and not set(DEV) & set(LOCKED)
assert set(DEV) <= set(SPLIT['SONICOM']['train']) | set(SPLIT['SONICOM']['val'])
assert set(LOCKED) <= set(SPLIT['SONICOM']['test'])
O = np.load(PATHS['fold_ref'], allow_pickle=True)   # 只用 dev / fold_of_subject
assert [str(s) for s in O['dev']] == DEV
PLAN = make_plan(len(DEV))
fold = np.full(len(DEV), -1)
for f, ((tr, va), _) in enumerate(PLAN):
    fold[np.unique(va // 2)] = f
assert np.array_equal(fold, O['fold_of_subject']), '折与 FFM_3_2_1 不同'
FOLD_ROW = np.repeat(fold, 2)
print(f'开发集 {len(DEV)} 人 / {2 * len(DEV)} 只耳，外层折与 FFM_3_2_1 相同；锁定 {len(LOCKED)} 人（本格不读）')

t0 = time.time()
X_meas, X_sim = load_rows('SONICOM', DEV), load_rows('SONICOM_SIM', DEV)
Zm, Zs = encode(X_meas, DM), encode(X_sim, DS)
GEO = geometry_blocks(DEV)
print(f'数据、latent、几何 {time.time() - t0:.0f}s | z {Zm.shape} | 几何块 ' + ', '.join(f'{k} {v.shape[1]}' for k, v in GEO.items()))
SOURCES = {'geometry': GEO, 'simulation': {'z_sim': Zs}}

# 训练折均值（实测、仿真）：每折一个
MU_M = np.stack([X_meas[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
MU_S = np.stack([X_sim[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
fold_mean = lambda MU: (lambda b0, b1: torch.from_numpy(MU[FOLD_ROW[b0:b1]]).to(device))
REF_PRED = {
    'training mean': fold_mean(MU_M),
    'raw simulation': lambda b0, b1: torch.from_numpy(X_sim[b0:b1]).to(device),
    'simulated deviation + measured mean': lambda b0, b1: torch.from_numpy(X_sim[b0:b1] - MU_S[FOLD_ROW[b0:b1]] + MU_M[FOLD_ROW[b0:b1]]).to(device),
    'AE reconstruction (reference)': ae_pred(X_meas, DM),
}
REF = {k: metrics(p, X_meas) for k, p in REF_PRED.items()}

In [ ]:
# ==========================================================================
# Cell 7 — 复现关卡：Task 1 的做法（全部开发行上标准化）必须复现 Task 1 的数
# ==========================================================================
TASK1 = {'geometry': (0.069445, 4.641051), 'simulation': (0.090528, 4.587035), 'geometry + simulation': (0.097233, 4.562941)}
TASK1_MEAN, TASK1_AE = 4.815128, 1.338633
got_mean, got_ae = summary(REF['training mean'])['pooled_rmse'], summary(REF['AE reconstruction (reference)'])['pooled_rmse']
print(f'均值 {got_mean:.4f}（Task 1 {TASK1_MEAN}）| AE {got_ae:.4f}（Task 1 {TASK1_AE}）')
assert abs(got_mean - TASK1_MEAN) < 0.005 and abs(got_ae - TASK1_AE) < 0.005
GATE = {}
for name, srcs in REGISTRATION['rows'].items():
    R = run_cv(kernel_fn([SOURCES[s] for s in srcs], fold_internal=False), Zm, PLAN)
    db = summary(metrics(decoder_pred(R['pred'], DM), X_meas))['pooled_rmse']
    GATE[name] = dict(r2=R['r2'], pooled_rmse=db)
    print(f"  {name:<24} CV-R² {R['r2']:+.4f}（Task 1 {TASK1[name][0]:+.4f}）| pooled {db:.4f}（Task 1 {TASK1[name][1]:.4f}）")
    assert abs(R['r2'] - TASK1[name][0]) < 0.002 and abs(db - TASK1[name][1]) < 0.005, f'{name}: 没有复现 Task 1 —— 停下来查'
print('✓ 复现了 Task 1；下面是每折内部标准化的正式版本')

In [ ]:
# ==========================================================================
# Cell 8 — 开发集正式结果（每折内部标准化）· 置换 · 配对比较 · 图 · 保存
# ==========================================================================
import matplotlib.pyplot as plt
t0 = time.time()
CV, PRED = {}, {}
for name, srcs in REGISTRATION['rows'].items():
    CV[name] = run_cv(kernel_fn([SOURCES[s] for s in srcs], fold_internal=True), Zm, PLAN)
    PRED[name] = metrics(decoder_pred(CV[name]['pred'], DM), X_meas)
    edge = sum(c in (0, len(ALPHA_REL) - 1) for c in CV[name]['alpha_idx'])
    print(f"  {name:<24} CV-R² {CV[name]['r2']:+.4f}（全局标准化 {GATE[name]['r2']:+.4f}）" + (f'  {edge} 折的 α 在网格边上' if edge else ''))
print(f'交叉验证 {time.time() - t0:.0f}s')

ROWS = {k: dict(r2=None, **summary(m)) for k, m in REF.items() if k != 'AE reconstruction (reference)'}
ROWS.update({k: dict(r2=CV[k]['r2'], **summary(PRED[k])) for k in CV})
ROWS['AE reconstruction (reference)'] = dict(r2=1.0, **summary(REF['AE reconstruction (reference)']))
show(f'开发集（{len(DEV)} 人，10 折交叉验证的折外预测；AE 是固定的）', ROWS)

# 置换：只对主行（几何）；按人打乱几何特征，再按同样的折内规则标准化
rng = np.random.default_rng(SEED + 1)
t0 = time.time()
null = []
for _ in range(N_PERM):
    p = rng.permutation(len(DEV)); r = np.array([2 * p[k] + s for k in range(len(DEV)) for s in (0, 1)])
    null.append(run_cv(kernel_fn([{k: v[r] for k, v in GEO.items()}], True), Zm, PLAN, keep_pred=False)['r2'])
null = np.array(null)
PERM = dict(p=float((1 + (null >= CV['geometry']['r2']).sum()) / (1 + N_PERM)), null_p95=float(np.percentile(null, 95)))
print(f"\n置换（几何 → z，{N_PERM} 次，{time.time() - t0:.0f}s）：p = {PERM['p']:.3f}，零分布 95 分位 {PERM['null_p95']:+.4f}")

# 按人配对：每行对均值；几何 + 仿真对几何
S_REF = per_subject(REF['training mean']); S = {k: per_subject(PRED[k]) for k in CV}
S_SIM = {k: per_subject(REF[k]) for k in ('raw simulation', 'simulated deviation + measured mean')}
PAIRED = {}
for key in ('lsd_el45', 'mae_el45', 'lsd_full', 'rmse_all'):
    for k in list(CV) + list(S_SIM):
        a = S[k][key] if k in S else S_SIM[k][key]
        PAIRED[f'{k} − mean · {key}'] = paired(a, S_REF[key], rng)
    PAIRED[f'geometry + simulation − geometry · {key}'] = paired(S['geometry + simulation'][key], S['geometry'][key], rng)
print('\n按人配对（负 = 前者更好；95% CI 来自 2000 次按人 bootstrap；p 是单侧配对 t 检验）')
for k, v in PAIRED.items():
    if k.endswith(('lsd_el45', 'mae_el45')):
        print(f"  {k:<58} {v['diff']:+.3f} dB [{v['lo']:+.3f}, {v['hi']:+.3f}]  p = {v['t_p_one_sided']:.2g}")

# 逐频率误差曲线（S2RNF 图 3 的样子）：每个 bin 在 ≥ −45° 的方向上取 RMS，再对耳和人平均
fig, ax = plt.subplots(figsize=(9, 4.2))
for k, c in (('training mean', 'k'), ('simulated deviation + measured mean', 'C7'), ('geometry', 'C0'),
             ('simulation', 'C1'), ('geometry + simulation', 'C3'), ('AE reconstruction (reference)', 'C2')):
    ax.plot(FREQS[EVAL_BINS] / 1000, ROWS[k]['curve_el45'], c=c, lw=1.6, label=k)
ax.set(xlabel='frequency (kHz)', ylabel='RMS error over directions (dB)', title='Development set, elevation ≥ −45°')
ax.grid(alpha=.3); ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(f'{OUT}/dev_error_vs_frequency.png', dpi=140); plt.show()
print(f"（原始仿真没画：LSD {ROWS['raw simulation']['lsd_el45']:.2f} dB，会把纵轴压扁）")

DEV_RES = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='mesh2z/FFM_3_8_Mesh2Z_48k.ipynb', registration=REGISTRATION,
               n_subjects=len(DEV), gate_task1=GATE, rows=ROWS, alpha_idx={k: CV[k]['alpha_idx'] for k in CV},
               permutation_geometry=PERM, paired=PAIRED, masuyama_table1=MASUYAMA)
json.dump(DEV_RES, open(f'{OUT}/dev_results.json', 'w'), indent=1, default=float)
np.savez_compressed(f'{OUT}/dev_oof.npz', dev=np.array(DEV), fold=fold,
                    **{f'zpred_{k.replace(" ", "_").replace("+", "plus")}': CV[k]['pred'].astype(np.float32) for k in CV},
                    **{f'subject_{k.replace(" ", "_").replace("+", "plus")}_{m}': v for k, d in {**S, 'mean': S_REF}.items() for m, v in d.items()})
print(f'\n-> {OUT}/dev_results.json, dev_oof.npz, dev_error_vs_frequency.png')

In [ ]:
# ==========================================================================
# Cell 9 — ★ TEST：36 个锁定受试者，只运行一次
#   登记（Cell 1）写进 PROJECT_LOG 之后才改 RUN_TEST = True。α 按同样规则在全部 164 人上选，均值也只来自这 164 人。
#   另算"协议校准"：在 S2RNF 的 160 / 25 划分上，按我们的口径算均值基线和原始仿真（不涉及任何模型）。
# ==========================================================================
S2RNF_VAL = [154, 222, 40, 244, 176, 285, 41, 208, 257, 218, 38, 130, 106, 103, 155]
S2RNF_TEST = [219, 320, 92, 101, 34, 243, 105, 309, 280, 315, 48, 71, 206, 113, 88, 112, 45, 28, 14, 142, 82, 319, 157, 58, 266]
TEST = None
if not RUN_TEST:
    print('RUN_TEST = False —— 测试集未读取')
else:
    used = [p for p in TEST_MARKS if os.path.exists(p)]
    assert not used, f'锁定测试集已经用过：{used}。不要再评估第二次。'
    rng = np.random.default_rng(SEED + 2)
    Xt_meas, Xt_sim = load_rows('SONICOM', LOCKED), load_rows('SONICOM_SIM', LOCKED)
    Zt_meas, Zt_sim = encode(Xt_meas, DM), encode(Xt_sim, DS)
    GEO_T = geometry_blocks(LOCKED)
    nd, na = 2 * len(DEV), 2 * len(DEV) + 2 * len(LOCKED)
    SRC_ALL = {'geometry': {k: np.concatenate([GEO[k], GEO_T[k]]) for k in GEO},
               'simulation': {'z_sim': np.concatenate([Zs, Zt_sim])}}
    MU_M_DEV = X_meas.astype(np.float64).mean(0).astype(np.float32)
    MU_S_DEV = X_sim.astype(np.float64).mean(0).astype(np.float32)
    T_REF = {
        'training mean': metrics(lambda b0, b1: torch.from_numpy(np.broadcast_to(MU_M_DEV, (b1 - b0,) + MU_M_DEV.shape).copy()).to(device), Xt_meas),
        'raw simulation': metrics(lambda b0, b1: torch.from_numpy(Xt_sim[b0:b1]).to(device), Xt_meas),
        'simulated deviation + measured mean': metrics(lambda b0, b1: torch.from_numpy(Xt_sim[b0:b1] - MU_S_DEV + MU_M_DEV).to(device), Xt_meas),
        'AE reconstruction (reference)': metrics(ae_pred(Xt_meas, DM), Xt_meas),
    }
    T_PRED, T_ALPHA, T_R2 = {}, {}, {}
    sst = ((Zt_meas - Zm.mean(0)) ** 2).sum()
    for name, srcs in REGISTRATION['rows'].items():
        Zp, T_ALPHA[name] = fit_predict(kernel_fn([SRC_ALL[s] for s in srcs], True), Zm, nd, na)
        T_R2[name] = float(1 - ((Zp - Zt_meas) ** 2).sum() / sst)
        T_PRED[name] = metrics(decoder_pred(Zp, DM), Xt_meas)
    TROWS = {k: dict(r2=None, **summary(m)) for k, m in T_REF.items() if k != 'AE reconstruction (reference)'}
    TROWS.update({k: dict(r2=T_R2[k], **summary(T_PRED[k])) for k in T_PRED})
    TROWS['AE reconstruction (reference)'] = dict(r2=1.0, **summary(T_REF['AE reconstruction (reference)']))
    show(f'TEST（{len(LOCKED)} 人，第一次也是唯一一次）', TROWS)
    TS_REF = per_subject(T_REF['training mean']); TS = {k: per_subject(T_PRED[k]) for k in T_PRED}
    TPAIRED = {}
    for key in ('lsd_el45', 'mae_el45', 'lsd_full', 'rmse_all'):
        for k in TS:
            TPAIRED[f'{k} − mean · {key}'] = paired(TS[k][key], TS_REF[key], rng)
        TPAIRED[f'geometry + simulation − geometry · {key}'] = paired(TS['geometry + simulation'][key], TS['geometry'][key], rng)
    print('\n按人配对（负 = 前者更好）')
    for k, v in TPAIRED.items():
        if k.endswith(('lsd_el45', 'mae_el45')):
            print(f"  {k:<58} {v['diff']:+.3f} dB [{v['lo']:+.3f}, {v['hi']:+.3f}]  p = {v['t_p_one_sided']:.2g}")

    # 协议校准：S2RNF 的划分（同样这 200 人），我们的数据和口径，只有均值和原始仿真
    ALL = DEV + LOCKED
    num = lambda s: int(s[-4:])
    XA_m, XA_s = np.concatenate([X_meas, Xt_meas]), np.concatenate([X_sim, Xt_sim])
    tr_rows = np.array([2 * i + e for i, s in enumerate(ALL) if num(s) not in S2RNF_VAL + S2RNF_TEST for e in (0, 1)])
    te_rows = np.array([2 * i + e for i, s in enumerate(ALL) if num(s) in S2RNF_TEST for e in (0, 1)])
    assert len(tr_rows) == 320 and len(te_rows) == 50, (len(tr_rows), len(te_rows))
    mu = XA_m[tr_rows].astype(np.float64).mean(0).astype(np.float32)
    Xte = XA_m[te_rows]
    CAL = {'training mean (their 160)': summary(metrics(lambda b0, b1: torch.from_numpy(np.broadcast_to(mu, (b1 - b0,) + mu.shape).copy()).to(device), Xte)),
           'raw simulation': summary(metrics(lambda b0, b1: torch.from_numpy(XA_s[te_rows][b0:b1]).to(device), Xte))}
    print('\n协议校准：S2RNF 的 25 个测试受试者，我们的数据与口径（他们：Avg. HRTF LSD 5.05 / MAE 3.78；Mesh2HRTF 10.53 / 7.50）')
    for k, r in CAL.items():
        print(f"  {k:<28} LSD≥−45° {r['lsd_el45']:.3f} | LSD 全球 {r['lsd_full']:.3f} | MAE≥−45° {r['mae_el45']:.3f}")

    TEST = dict(created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='FFM_3_8', registration=REGISTRATION, n_subjects=len(LOCKED),
                rows=TROWS, alpha_idx=T_ALPHA, paired=TPAIRED, calibration_s2rnf_split=CAL, masuyama_table1=MASUYAMA,
                note='also reads FFM48k_measim on 36 of its 60 SONICOM test subjects')
    for p in (TEST_MARKS[0], TEST_MARKS[3]):
        os.makedirs(os.path.dirname(p), exist_ok=True); json.dump(TEST, open(p, 'w'), indent=1, default=float)
    json.dump(TEST, open(f'{OUT}/test_results.json', 'w'), indent=1, default=float)
    print(f'\n已记录 -> {OUT}/test_results.json；锁定测试集现在已经用过了（FFM_3_5 / 3_6 / 3_7 的测试格也会拒绝）')

## 跑完之后

1. 开发集：把 Cell 7 和 Cell 8 的输出贴回来（或下载 `mesh2z_48k_v1/dev_results.json` 到 `hrtf-GitHub/results/`）。
   先看每折内部标准化和全局标准化差多少、三行相对均值各多少、置换 p。
2. 决定是否动用测试集：确认 Cell 1 的登记，写进 PROJECT_LOG，再把 `RUN_TEST` 改成 True 跑 Cell 9（只一次）。